In [0]:
USE CATALOG projectcatalog

In [0]:
use schema Gold_Schema

In [0]:
-- 2. Preview Silver source
SELECT * FROM sliverschemasales.tblprjregionssilver ORDER BY region_id;

region_id,region,silver_load_created_date,source_file,data_layer
R01,East,2026-08-11T18:32:08.835Z,RegionsFile.parquet,SILVER
R02,West,2026-08-11T18:32:08.835Z,RegionsFile.parquet,SILVER
R03,North,2026-08-11T18:32:08.835Z,RegionsFile.parquet,SILVER
R04,South,2026-08-11T18:32:08.835Z,RegionsFile.parquet,SILVER


In [0]:
-- 3. Create Gold region dimension
CREATE TABLE IF NOT EXISTS  dim_region
(
    region_key             BIGINT,
    region_id              STRING,
    region_name            STRING,
    source_file            STRING,
    silver_load_datetime   TIMESTAMP,
    gold_created_datetime  TIMESTAMP,
    gold_updated_datetime  TIMESTAMP
)
USING DELTA;


In [0]:
-- 4. Prepare latest Silver record for each region
CREATE OR REPLACE TEMP VIEW regions_gold_source AS
SELECT
    XXHASH64(region_id) AS region_key,
    region_id,
    region AS region_name,
    source_file,
    silver_load_created_date AS silver_load_datetime,
    CURRENT_TIMESTAMP() AS load_datetime
FROM
(
    SELECT
        *,
        ROW_NUMBER() OVER
        (
            PARTITION BY region_id
            ORDER BY silver_load_created_date DESC
        ) AS row_num
    FROM sliverschemasales.tblprjregionssilver
) source
WHERE row_num = 1;

In [0]:
select * from regions_gold_source

region_key,region_id,region_name,source_file,silver_load_datetime,load_datetime
-795916668163233541,R01,East,RegionsFile.parquet,2026-08-11T18:32:08.835Z,2026-09-08T10:40:20.240Z
7308453538943115827,R02,West,RegionsFile.parquet,2026-08-11T18:32:08.835Z,2026-09-08T10:40:20.240Z
753262185817616840,R03,North,RegionsFile.parquet,2026-08-11T18:32:08.835Z,2026-09-08T10:40:20.240Z
5737401384420165154,R04,South,RegionsFile.parquet,2026-08-11T18:32:08.835Z,2026-09-08T10:40:20.240Z


In [0]:
-- 5. SCD Type 1 MERGE
-- New region_id: INSERT
-- Changed region name/source: UPDATE
-- Unchanged row: no action

MERGE INTO    dim_region AS target 
USING regions_gold_source AS source
ON target.region_id = source.region_id
WHEN MATCHED AND
(
       NOT (target.region_name <=> source.region_name)
    OR NOT (target.source_file <=> source.source_file)
)
THEN UPDATE SET
    target.region_name            = source.region_name,
    target.source_file            = source.source_file,
    target.silver_load_datetime   = source.silver_load_datetime,
    target.gold_updated_datetime  = source.load_datetime
WHEN NOT MATCHED
THEN INSERT
(
    region_key,region_id,region_name,source_file,silver_load_datetime,gold_created_datetime, gold_updated_datetime
)
VALUES
(
    source.region_key,
    source.region_id,
    source.region_name,
    source.source_file,
    source.silver_load_datetime,
    source.load_datetime,
    source.load_datetime
);


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,0,0,4


In [0]:
-- 6. Preview Gold dimension
SELECT * FROM dim_region ORDER BY region_id;

region_key,region_id,region_name,source_file,silver_load_datetime,gold_created_datetime,gold_updated_datetime
-795916668163233541,R01,East,RegionsFile.parquet,2026-08-11T18:32:08.835Z,2026-09-08T10:40:50.501Z,2026-09-08T10:40:50.501Z
7308453538943115827,R02,West,RegionsFile.parquet,2026-08-11T18:32:08.835Z,2026-09-08T10:40:50.501Z,2026-09-08T10:40:50.501Z
753262185817616840,R03,North,RegionsFile.parquet,2026-08-11T18:32:08.835Z,2026-09-08T10:40:50.501Z,2026-09-08T10:40:50.501Z
5737401384420165154,R04,South,RegionsFile.parquet,2026-08-11T18:32:08.835Z,2026-09-08T10:40:50.501Z,2026-09-08T10:40:50.501Z


In [0]:
-- 7. Check duplicate business keys
-- Expected result: zero rows
SELECT region_id, COUNT(*) AS duplicate_count FROM dim_region GROUP BY region_id HAVING COUNT(*) > 1;


region_id,duplicate_count


In [0]:
-- 8. Check duplicate surrogate keys 
SELECT region_key, COUNT(*) AS duplicate_count
FROM    dim_region GROUP BY region_key HAVING COUNT(*) > 1;

region_key,duplicate_count


In [0]:
-- 9. Silver-to-Gold reconciliation
SELECT
    (
        SELECT COUNT(DISTINCT region_id)
        FROM sliverschemasales.tblprjregionssilver
    ) AS silver_region_count,
    (
        SELECT COUNT(*)
        FROM dim_region
    ) AS gold_region_count;

silver_region_count,gold_region_count
4,4


In [0]:
-- 10. Identify records changed after initial insert
SELECT * FROM dim_region WHERE gold_updated_datetime > gold_created_datetime 
ORDER BY gold_updated_datetime DESC;


region_key,region_id,region_name,source_file,silver_load_datetime,gold_created_datetime,gold_updated_datetime


In [0]:
-- 11. Simple reporting view
CREATE OR REPLACE VIEW vw_regions AS
SELECT region_key, region_id, region_name FROM dim_region;


In [0]:
SELECT * FROM vw_regions ORDER BY region_id;

region_key,region_id,region_name
-795916668163233541,R01,East
7308453538943115827,R02,West
753262185817616840,R03,North
5737401384420165154,R04,South


In [0]:
-- 12. Optional optimization
OPTIMIZE dim_region ZORDER BY (region_id);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2606), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1788864205943, 1788864206712, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"
